# Automated Analytics Report v2 (Case Study Upgrade)

This upgraded notebook keeps the original notebook untouched and runs a recruiter-grade pipeline on a stronger business dataset.

What is upgraded:
- Reproducible UCI Online Retail dataset loading
- Business feature engineering (returns/cancellations, revenue, cohorts, RFM-ready fields)
- Hybrid routing (OpenAI for complex tasks + existing models for baseline/fallback)
- Deterministic execution and grounded reporting
- HTML, PDF, executive summary, technical appendix, and benchmark output


In [ ]:
# Install dependencies (run once per environment)
# !pip install -q -e .[dev]


## 0) API keys (environment variables only)

Set keys in your shell/session before running:

- `CYVERSE_API_KEY` (required)
- `OPENAI_API_KEY` (recommended for complex planning/critic tasks)

Do not paste keys directly into notebook cells.


In [ ]:
import os
required = ["CYVERSE_API_KEY"]
optional = ["OPENAI_API_KEY"]
for env in required:
    if not os.getenv(env):
        raise RuntimeError(f"Missing required env var: {env}")
print("Required keys are set.")
for env in optional:
    print(f"{env} set:", bool(os.getenv(env)))


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from automated_llm_pred.config import PipelineConfig
from automated_llm_pred.constants import DEFAULT_PLOT_GOALS, DEFAULT_QUESTIONS
from automated_llm_pred.data import fetch_uci_online_retail, prepare_online_retail_business_view
from automated_llm_pred.pipeline import run_case_study


## 1) Load and prepare upgraded business dataset


In [ ]:
raw_df = fetch_uci_online_retail(cache_dir=".cache")
df = prepare_online_retail_business_view(raw_df)

print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {len(df.columns)}")
print("Columns:", list(df.columns)[:20])
df.head()


## 2) Enter your questions and plot goals

Type your own questions/goals.

If you leave both sections blank, the notebook automatically uses curated default case-study prompts.


In [ ]:
questions = []
while True:
    q = input("Enter a QUESTION (blank to finish): ").strip()
    if not q:
        break
    questions.append(q)
plot_goals = []
while True:
    g = input("Enter a PLOT GOAL (blank to finish): ").strip()
    if not g:
        break
    plot_goals.append(g)
if not questions:
    questions = DEFAULT_QUESTIONS.copy()
    print("No custom questions entered. Using default case-study questions.")
if not plot_goals:
    plot_goals = DEFAULT_PLOT_GOALS.copy()
    print("No custom plot goals entered. Using default case-study plot goals.")
print("Total questions:", len(questions))
print("Total plot goals:", len(plot_goals))


## 3) Run case study pipeline and export outputs


In [ ]:
out_dir = "report_output_v2"
title = "Automated Business Analytics Case Study (LLM + Deterministic Execution)"

paths = run_case_study(
    df=df,
    out_dir=out_dir,
    title=title,
    questions=questions,
    plot_goals=plot_goals,
    cfg=PipelineConfig(),
    use_defaults_when_empty=True,
)

paths


In [ ]:
from pathlib import Path
for name, p in paths.items():
    print(f"{name}: {Path(p).resolve()}")


## 4) CLI equivalent

Run the same workflow outside notebook:

```bash
run-case-study --out-dir report_output_v2
```

Optional: provide `--questions-file` and `--plot-goals-file` (JSON list or line-separated text).
